# Convex Optimization — Interactive Theory

> *"The great watershed in optimization isn't between linear and nonlinear — it's between convex and nonconvex."* — R. Tyrrell Rockafellar

This notebook provides interactive demonstrations of convexity, duality, and optimization problem classes. Every major result from the notes is verified numerically.

In [ ]:
import numpy as np
import scipy.linalg as la
from scipy.optimize import minimize

try:
    import matplotlib.pyplot as plt
    import matplotlib
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [10, 6]
    plt.rcParams['figure.dpi'] = 120
    plt.rcParams['font.size'] = 13
    plt.rcParams['axes.titlesize'] = 15
    plt.rcParams['axes.labelsize'] = 13
    plt.rcParams['axes.spines.top'] = False
    plt.rcParams['axes.spines.right'] = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid', palette='colorblind')
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

COLORS = {
    'primary':   '#0077BB',
    'secondary': '#EE7733',
    'tertiary':  '#009988',
    'error':     '#CC3311',
    'neutral':   '#555555',
    'highlight': '#EE3377',
}

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

print('Setup complete.')

---

## 1. Intuition: Convex vs Nonconvex Landscapes

In [ ]:
# === 1.1 Convex vs Nonconvex Landscape Visualization ===

x = np.linspace(-3, 3, 300)

# Convex function: quadratic
f_convex = x**2

# Nonconvex function: multiple local minima
f_nonconvex = x**4 - 4*x**2 + 0.5*x + 4

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(x, f_convex, color=COLORS['primary'], linewidth=2.5)
    axes[0].scatter([0], [0], color=COLORS['error'], s=100, zorder=5, label='Global min')
    axes[0].set_title('Convex function: $f(x) = x^2$')
    axes[0].set_xlabel('$x$')
    axes[0].set_ylabel('$f(x)$')
    axes[0].legend()
    
    axes[1].plot(x, f_nonconvex, color=COLORS['secondary'], linewidth=2.5)
    # Find local minima numerically
    from scipy.signal import argrelmin
    mins = argrelmin(f_nonconvex, order=10)[0]
    axes[1].scatter(x[mins], f_nonconvex[mins], color=COLORS['error'], s=100, zorder=5,
                    label=f'{len(mins)} local minima')
    axes[1].set_title('Nonconvex: $f(x) = x^4 - 4x^2 + 0.5x + 4$')
    axes[1].set_xlabel('$x$')
    axes[1].set_ylabel('$f(x)$')
    axes[1].legend()
    
    fig.suptitle('Convex vs Nonconvex Landscapes', fontsize=15, y=1.02)
    fig.tight_layout()
    plt.show()

print('Convex function has UNIQUE minimum at x=0')
print(f'Nonconvex function has {len(mins)} local minima')

In [ ]:
# === 1.3 Key Dates in Convex Optimization ===

timeline = [
    (1847, 'Cauchy', 'First gradient descent algorithm'),
    (1947, 'Dantzig', 'Simplex method for LP'),
    (1951, 'Kuhn-Tucker', 'KKT conditions'),
    (1970, 'Rockafellar', 'Convex Analysis textbook'),
    (1984, 'Karmarkar', 'Interior-point methods'),
    (1995, 'Vapnik', 'SVM (convex QP for ML)'),
    (2004, 'Boyd-Vandenberghe', 'Convex Optimization textbook'),
    (2022, 'Hu et al.', 'LoRA (low-rank convex penalty)'),
]

print('Convex Optimization Timeline')
print('=' * 60)
for year, who, what in timeline:
    print(f'  {year}  {who:20s}  {what}')
print('=' * 60)

---

## 2. Convex Sets

In [ ]:
# === 2.1 Verifying Convex Sets ===

def is_in_ball(x, center, radius):
    """Check if x is in the Euclidean ball B(center, radius)."""
    return np.linalg.norm(x - center) <= radius + 1e-10

def is_in_simplex(p):
    """Check if p is in the probability simplex."""
    return np.all(p >= -1e-10) and abs(np.sum(p) - 1.0) < 1e-10

# Test convexity: for random pairs in the set, check midpoint
n_tests = 1000

# Test 1: Euclidean ball
center = np.array([1.0, 2.0])
radius = 3.0
pass_count = 0
for _ in range(n_tests):
    # Generate random points in the ball
    d = np.random.randn(2)
    x = center + radius * d / np.linalg.norm(d) * np.random.uniform(0, 1)
    d = np.random.randn(2)
    y = center + radius * d / np.linalg.norm(d) * np.random.uniform(0, 1)
    theta = np.random.uniform(0, 1)
    midpoint = theta * x + (1 - theta) * y
    if is_in_ball(midpoint, center, radius):
        pass_count += 1
print(f'Ball convexity: {pass_count}/{n_tests} midpoints in set')
ok1 = pass_count == n_tests
print(f"{'PASS' if ok1 else 'FAIL'} — Euclidean ball is convex")

# Test 2: Probability simplex
pass_count = 0
for _ in range(n_tests):
    p1 = np.random.dirichlet(np.ones(5))
    p2 = np.random.dirichlet(np.ones(5))
    theta = np.random.uniform(0, 1)
    midpoint = theta * p1 + (1 - theta) * p2
    if is_in_simplex(midpoint):
        pass_count += 1
print(f'\nSimplex convexity: {pass_count}/{n_tests} midpoints in set')
ok2 = pass_count == n_tests
print(f"{'PASS' if ok2 else 'FAIL'} — Probability simplex is convex")

In [ ]:
# === 2.1 Non-Example: Unit Sphere (boundary only) ===

# The unit sphere {x : ||x|| = 1} is NOT convex
x = np.array([1.0, 0.0])
y = np.array([-1.0, 0.0])
midpoint = 0.5 * x + 0.5 * y

print(f'x = {x}, ||x|| = {np.linalg.norm(x):.1f}')
print(f'y = {y}, ||y|| = {np.linalg.norm(y):.1f}')
print(f'midpoint = {midpoint}, ||midpoint|| = {np.linalg.norm(midpoint):.1f}')
print(f'Midpoint on unit sphere? {np.isclose(np.linalg.norm(midpoint), 1.0)}')
print()
ok = not np.isclose(np.linalg.norm(midpoint), 1.0)
print(f"{'PASS' if ok else 'FAIL'} — Unit sphere is NOT convex (midpoint has norm 0, not 1)")

In [ ]:
# === 2.1 Visualize Convex vs Non-Convex Sets ===

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Ball (convex)
    theta = np.linspace(0, 2*np.pi, 100)
    axes[0].fill(2*np.cos(theta), 2*np.sin(theta), alpha=0.3, color=COLORS['primary'])
    axes[0].plot(2*np.cos(theta), 2*np.sin(theta), color=COLORS['primary'], linewidth=2)
    axes[0].plot([1, -1], [1.5, -0.5], 'o--', color=COLORS['error'], linewidth=2)
    axes[0].set_title('Ball (convex)')
    axes[0].set_aspect('equal')
    axes[0].set_xlabel('$x_1$')
    axes[0].set_ylabel('$x_2$')
    
    # Simplex (convex) in 2D
    tri = plt.Polygon([[0, 0], [1, 0], [0, 1]], alpha=0.3, color=COLORS['tertiary'])
    axes[1].add_patch(tri)
    axes[1].plot([0, 1, 0, 0], [0, 0, 1, 0], color=COLORS['tertiary'], linewidth=2)
    axes[1].plot([0.2, 0.5], [0.3, 0.1], 'o--', color=COLORS['error'], linewidth=2)
    axes[1].set_title('Simplex (convex)')
    axes[1].set_aspect('equal')
    axes[1].set_xlim(-0.2, 1.2)
    axes[1].set_ylim(-0.2, 1.2)
    axes[1].set_xlabel('$p_1$')
    axes[1].set_ylabel('$p_2$')
    
    # Unit sphere boundary (NOT convex)
    axes[2].plot(np.cos(theta), np.sin(theta), color=COLORS['secondary'], linewidth=2)
    axes[2].plot([1, -1], [0, 0], 'o--', color=COLORS['error'], linewidth=2)
    axes[2].plot(0, 0, 's', color=COLORS['highlight'], markersize=10, label='Midpoint (NOT on sphere)')
    axes[2].set_title('Unit sphere boundary (NOT convex)')
    axes[2].set_aspect('equal')
    axes[2].legend(fontsize=10)
    axes[2].set_xlabel('$x_1$')
    axes[2].set_ylabel('$x_2$')
    
    fig.tight_layout()
    plt.show()

print('Line segment between two points: inside ball/simplex, outside sphere boundary')

---

### 2.3 Separating Hyperplane Theorem

In [ ]:
# === 2.3 Separating Hyperplane Visualization ===

if HAS_MPL:
    np.random.seed(42)
    # Two disjoint convex sets (clusters)
    C1 = np.random.randn(30, 2) * 0.5 + np.array([-2, 0])
    C2 = np.random.randn(30, 2) * 0.5 + np.array([2, 0])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(C1[:, 0], C1[:, 1], color=COLORS['primary'], s=60, label='$C_1$', alpha=0.7)
    ax.scatter(C2[:, 0], C2[:, 1], color=COLORS['secondary'], s=60, label='$C_2$', alpha=0.7)
    
    # Separating hyperplane (vertical line at x=0)
    ax.axvline(0, color=COLORS['neutral'], linestyle='--', linewidth=2,
               label='Separating hyperplane')
    ax.fill_betweenx([-2, 2], -4, 0, alpha=0.05, color=COLORS['primary'])
    ax.fill_betweenx([-2, 2], 0, 4, alpha=0.05, color=COLORS['secondary'])
    
    ax.set_title('Separating Hyperplane Theorem')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.legend()
    ax.set_xlim(-4, 4)
    ax.set_ylim(-2, 2)
    fig.tight_layout()
    plt.show()

print('Disjoint convex sets can always be separated by a hyperplane')

---

### 2.4 The PSD Cone

In [ ]:
# === 2.4 PSD Cone is Convex ===

def is_psd(A, tol=1e-10):
    """Check if symmetric matrix A is PSD."""
    eigvals = np.linalg.eigvalsh(A)
    return np.all(eigvals >= -tol)

# Generate random PSD matrices
n = 4
n_tests = 500
pass_count = 0

for _ in range(n_tests):
    # Random PSD matrix: A = B^T B
    B1 = np.random.randn(n, n)
    A1 = B1.T @ B1
    B2 = np.random.randn(n, n)
    A2 = B2.T @ B2
    
    theta = np.random.uniform(0, 1)
    A_mix = theta * A1 + (1 - theta) * A2
    
    if is_psd(A_mix):
        pass_count += 1

print(f'PSD cone convexity: {pass_count}/{n_tests} convex combinations are PSD')
ok = pass_count == n_tests
print(f"{'PASS' if ok else 'FAIL'} — PSD cone is convex")

# Show eigenvalues of a convex combination
B1 = np.random.randn(3, 3)
A1 = B1.T @ B1
B2 = np.random.randn(3, 3)
A2 = B2.T @ B2
A_half = 0.5 * A1 + 0.5 * A2
print(f'\nEigenvalues of A1: {np.sort(np.linalg.eigvalsh(A1))}')
print(f'Eigenvalues of A2: {np.sort(np.linalg.eigvalsh(A2))}')
print(f'Eigenvalues of 0.5*A1 + 0.5*A2: {np.sort(np.linalg.eigvalsh(A_half))}')
print('All eigenvalues non-negative ✓')

---

## 3. Convex Functions

In [ ]:
# === 3.1 First-Order Condition: f(y) >= f(x) + grad(x)^T (y-x) ===

# Test on f(x) = x^2, f'(x) = 2x
def f(x): return x**2
def grad_f(x): return 2*x

n_tests = 1000
pass_count = 0
for _ in range(n_tests):
    x = np.random.uniform(-5, 5)
    y = np.random.uniform(-5, 5)
    # First-order condition: f(y) >= f(x) + f'(x)*(y-x)
    if f(y) >= f(x) + grad_f(x) * (y - x) - 1e-10:
        pass_count += 1

print(f'First-order condition holds: {pass_count}/{n_tests} pairs')
ok = pass_count == n_tests
print(f"{'PASS' if ok else 'FAIL'} — f(x)=x^2 satisfies first-order characterization")

# Visualize the tangent as global underestimator
if HAS_MPL:
    x_pts = np.linspace(-3, 3, 300)
    x0 = 1.0
    tangent = f(x0) + grad_f(x0) * (x_pts - x0)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(x_pts, f(x_pts), color=COLORS['primary'], linewidth=2.5, label='$f(x) = x^2$')
    ax.plot(x_pts, tangent, color=COLORS['secondary'], linewidth=2,
            linestyle='--', label=f'Tangent at $x_0={x0}$')
    ax.scatter([x0], [f(x0)], color=COLORS['error'], s=100, zorder=5)
    ax.fill_between(x_pts, tangent, f(x_pts), alpha=0.15, color=COLORS['primary'],
                    label='$f(y) - $ tangent $\geq 0$')
    ax.set_title('First-Order Condition: Tangent is Global Underestimator')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$f(x)$')
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
# === 3.2 Second-Order Condition: Hessian PSD ===

def check_convexity_hessian(Q):
    """Check if quadratic f(x) = 0.5 x^T Q x is convex via eigenvalues."""
    eigvals = np.linalg.eigvalsh(Q)
    return eigvals, np.all(eigvals >= -1e-10)

# Example 1: Convex quadratic
Q1 = np.array([[4.0, 1.0], [1.0, 3.0]])
eigvals1, is_convex1 = check_convexity_hessian(Q1)
print('Q1 (should be convex):')
print(f'  Q1 = {Q1}')
print(f'  Eigenvalues: {eigvals1}')
print(f"  {'PASS' if is_convex1 else 'FAIL'} — {'Convex' if is_convex1 else 'Not convex'}")

# Example 2: Non-convex (indefinite)
Q2 = np.array([[1.0, 0.0], [0.0, -1.0]])
eigvals2, is_convex2 = check_convexity_hessian(Q2)
print('\nQ2 (should NOT be convex):')
print(f'  Q2 = {Q2}')
print(f'  Eigenvalues: {eigvals2}')
print(f"  {'PASS' if not is_convex2 else 'FAIL'} — {'Correctly detected as NOT convex' if not is_convex2 else 'Incorrectly convex'}")

# Example 3: Ridge regression Hessian
np.random.seed(42)
n, d, lam = 100, 5, 0.1
X = np.random.randn(n, d)
H_ols = X.T @ X
H_ridge = X.T @ X + lam * np.eye(d)
_, ols_convex = check_convexity_hessian(H_ols)
_, ridge_convex = check_convexity_hessian(H_ridge)
print(f'\nOLS Hessian X^T X: min eigenvalue = {np.linalg.eigvalsh(H_ols).min():.6f}')
print(f'Ridge Hessian X^T X + λI: min eigenvalue = {np.linalg.eigvalsh(H_ridge).min():.6f}')
print(f"{'PASS' if ridge_convex else 'FAIL'} — Ridge regression is strictly convex (λ={lam})")

In [ ]:
# === 3.3 Log-Sum-Exp: Key Function for Cross-Entropy ===

def log_sum_exp(z):
    """Numerically stable log-sum-exp."""
    m = np.max(z)
    return m + np.log(np.sum(np.exp(z - m)))

def softmax(z):
    z_shift = z - np.max(z)
    e = np.exp(z_shift)
    return e / e.sum()

def lse_hessian(z):
    """Hessian of log-sum-exp: diag(p) - p p^T"""
    p = softmax(z)
    return np.diag(p) - np.outer(p, p)

# Verify Hessian is PSD for random logit vectors
n_tests = 500
pass_count = 0
for _ in range(n_tests):
    z = np.random.randn(np.random.randint(2, 20))
    H = lse_hessian(z)
    eigvals = np.linalg.eigvalsh(H)
    if np.all(eigvals >= -1e-9):
        pass_count += 1

print(f'LSE Hessian PSD: {pass_count}/{n_tests} random logit vectors')
ok = pass_count == n_tests
print(f"{'PASS' if ok else 'FAIL'} — Log-sum-exp Hessian is always PSD")

# Cross-entropy loss is convex in logits
z = np.array([2.0, 1.0, 0.5, -0.5])
true_class = 0
loss = -z[true_class] + log_sum_exp(z)
H = lse_hessian(z)
print(f'\nCross-entropy loss = {loss:.4f}')
print(f'Hessian eigenvalues: {np.linalg.eigvalsh(H)}')
print('All >= 0: cross-entropy is convex in logits ✓')

In [ ]:
# === 3.1 Jensen's Inequality Demo ===

# f(E[X]) <= E[f(X)] for convex f
# Test with f(x) = x^2 and random variable X

np.random.seed(42)
n_samples = 10000

# Convex: f(x) = x^2, Jensen: E[X]^2 <= E[X^2]
X = np.random.randn(n_samples)
EX = np.mean(X)
EX2 = np.mean(X**2)
EX_sq = EX**2

print('Jensen inequality for f(x) = x^2:')
print(f'  E[X]^2 = {EX_sq:.6f}')
print(f'  E[X^2] = {EX2:.6f}')
print(f"  {'PASS' if EX_sq <= EX2 + 1e-10 else 'FAIL'} — f(E[X]) <= E[f(X)]")

# Concave: f(x) = log(x), Jensen: E[log(X)] <= log(E[X])
X_pos = np.abs(X) + 0.1
E_log_X = np.mean(np.log(X_pos))
log_EX = np.log(np.mean(X_pos))

print('\nJensen inequality for f(x) = log(x) (concave):')
print(f'  E[log(X)] = {E_log_X:.6f}')
print(f'  log(E[X]) = {log_EX:.6f}')
print(f"  {'PASS' if E_log_X <= log_EX + 1e-10 else 'FAIL'} — E[f(X)] <= f(E[X]) for concave f")

---

## 4. Strong Convexity and Smoothness

In [ ]:
# === 4.1-4.3 Strong Convexity, Smoothness, Condition Number ===

def convexity_params(Q):
    """Compute mu, L, kappa for quadratic f(x) = 0.5 x^T Q x."""
    eigvals = np.linalg.eigvalsh(Q)
    mu = eigvals.min()   # strong convexity constant
    L  = eigvals.max()   # smoothness constant
    kappa = L / mu if mu > 1e-12 else np.inf
    return mu, L, kappa

examples = [
    ('Diagonal Q=[4,1]', np.diag([4.0, 1.0])),
    ('Ridge n=100, d=5, lam=0.1', X.T @ X + 0.1 * np.eye(5) if 'X' in dir() else None),
]

np.random.seed(42)
X_data = np.random.randn(100, 5)

print(f'{"Example":<35} {"mu (str. cvx)":>15} {"L (smooth)":>12} {"kappa":>10}')
print('-' * 75)

Qs = [
    ('Diagonal [4, 1]',          np.diag([4.0, 1.0])),
    ('Diagonal [10, 0.01]',       np.diag([10.0, 0.01])),
    ('Ridge (lam=1.0)',           X_data.T @ X_data + 1.0 * np.eye(5)),
    ('Ridge (lam=0.01)',          X_data.T @ X_data + 0.01 * np.eye(5)),
    ('Ridge (lam=0.001)',         X_data.T @ X_data + 0.001 * np.eye(5)),
]

for name, Q in Qs:
    mu, L, kappa = convexity_params(Q)
    print(f'{name:<35} {mu:>15.4f} {L:>12.4f} {kappa:>10.1f}')

print('\nKey insight: larger lambda -> better conditioned (smaller kappa)')

In [ ]:
# === 4.2 Descent Lemma: f(x - eta*grad) <= f(x) - (eta/2)*||grad||^2 ===

# For f(x) = 0.5 x^T Q x with L = lambda_max(Q)
Q = np.diag([4.0, 1.0])
L = np.linalg.eigvalsh(Q).max()
eta = 1.0 / L  # optimal step size

def f_quad(x): return 0.5 * x @ Q @ x
def grad_quad(x): return Q @ x

np.random.seed(42)
n_tests = 1000
pass_count = 0
for _ in range(n_tests):
    x = np.random.randn(2) * 3
    g = grad_quad(x)
    x_new = x - eta * g
    # Descent lemma: f(x_new) <= f(x) - (eta/2) * ||g||^2
    rhs = f_quad(x) - (eta / 2) * np.dot(g, g)
    if f_quad(x_new) <= rhs + 1e-10:
        pass_count += 1

print(f'Descent lemma holds: {pass_count}/{n_tests} steps')
ok = pass_count == n_tests
print(f"{'PASS' if ok else 'FAIL'} — Descent lemma verified for eta = 1/L")

# Show convergence of GD with optimal step size
x = np.array([3.0, 2.0])
history = [f_quad(x)]
for _ in range(30):
    x = x - eta * grad_quad(x)
    history.append(f_quad(x))

mu = np.linalg.eigvalsh(Q).min()
kappa = L / mu
print(f'\nL={L}, mu={mu}, kappa={kappa:.1f}')
print(f'After 30 steps: f = {history[-1]:.2e} (started at {history[0]:.2f})')

if HAS_MPL:
    theory_rate = [(1 - 1/kappa)**t * history[0] for t in range(31)]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.semilogy(history, color=COLORS['primary'], linewidth=2, label='GD loss')
    ax.semilogy(theory_rate, color=COLORS['secondary'], linewidth=2,
               linestyle='--', label=f'$(1-1/\\kappa)^t$ bound, $\\kappa={kappa:.0f}$')
    ax.set_title('GD Convergence on Quadratic (log scale)')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('$f(x_t) - f^*$')
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
# === 4.3 Condition Number vs Convergence Speed ===

def run_gd(Q, n_steps=100):
    """Run GD on f(x) = 0.5 x^T Q x from x0 = [1,1,...,1]."""
    L = np.linalg.eigvalsh(Q).max()
    eta = 1.0 / L
    x = np.ones(Q.shape[0])
    losses = [0.5 * x @ Q @ x]
    for _ in range(n_steps):
        x = x - eta * Q @ x
        losses.append(0.5 * x @ Q @ x)
    return np.array(losses)

kappas = [1.0, 5.0, 25.0, 100.0]
n = 10

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 5))
    for kappa in kappas:
        # Build Q with eigenvalues 1 and kappa (2x2 case for illustration)
        Q = np.diag([kappa, 1.0])
        losses = run_gd(Q, n_steps=150)
        ax.semilogy(losses / losses[0], linewidth=2, label=f'$\\kappa={kappa:.0f}$')
    ax.set_title('GD Convergence vs Condition Number $\\kappa = L/\\mu$')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Relative loss $f(x_t)/f(x_0)$')
    ax.legend()
    fig.tight_layout()
    plt.show()

print('Steps to reach 1e-6 relative accuracy:')
for kappa in kappas:
    Q = np.diag([kappa, 1.0])
    losses = run_gd(Q, n_steps=5000)
    rel = losses / losses[0]
    idx = np.argmax(rel < 1e-6) if np.any(rel < 1e-6) else 5000
    print(f'  kappa={kappa:6.1f}: {idx} steps')

---

## 5. Convex Optimization Problems (LP, QP, SOCP, SDP)

In [ ]:
# === 5.2 LP: L1-Minimization for Sparse Recovery ===

from scipy.optimize import linprog

# Compressed sensing: recover sparse x from measurements b = Ax
# min ||x||_1 s.t. Ax = b  <=>  LP via u,v >= 0, x = u - v
np.random.seed(42)
m, n_feat = 30, 50   # underdetermined: 30 equations, 50 unknowns
k_sparse = 5         # true signal has only 5 non-zeros

A_cs = np.random.randn(m, n_feat)
# True sparse signal
x_true = np.zeros(n_feat)
support = np.random.choice(n_feat, k_sparse, replace=False)
x_true[support] = np.random.randn(k_sparse)
b_cs = A_cs @ x_true

# LP formulation: min 1^T u + 1^T v s.t. A(u-v)=b, u,v >= 0
c_lp = np.ones(2 * n_feat)   # objective: sum u_i + sum v_i
A_eq = np.hstack([A_cs, -A_cs])  # A(u-v) = b

result = linprog(c_lp, A_eq=A_eq, b_eq=b_cs,
                 bounds=[(0, None)] * (2 * n_feat),
                 method='highs')

if result.success:
    uv = result.x
    x_recovered = uv[:n_feat] - uv[n_feat:]
    err = np.linalg.norm(x_recovered - x_true) / np.linalg.norm(x_true)
    sparsity = np.sum(np.abs(x_recovered) > 1e-4)
    print(f'LP solved: status={result.status}')
    print(f'Relative recovery error: {err:.2e}')
    print(f'Recovered sparsity: {sparsity} nonzeros (true: {k_sparse})')
    ok = err < 1e-4
    print(f"{'PASS' if ok else 'FAIL'} — L1 minimization recovers sparse signal")
else:
    print('LP solver failed:', result.message)

In [ ]:
# === 5.3 QP: Ridge Regression Closed Form vs Gradient Descent ===

np.random.seed(42)
n_samples, n_features = 50, 10
lam = 0.5

X_r = np.random.randn(n_samples, n_features)
w_true = np.random.randn(n_features)
y_r = X_r @ w_true + 0.1 * np.random.randn(n_samples)

# Closed-form Ridge: w* = (X^T X + lambda I)^{-1} X^T y
Q_ridge = X_r.T @ X_r + lam * np.eye(n_features)
w_closed = np.linalg.solve(Q_ridge, X_r.T @ y_r)

# GD: gradient of 0.5||Xw-y||^2 + 0.5*lam*||w||^2 = X^T(Xw-y) + lam*w
L_gd = np.linalg.eigvalsh(Q_ridge).max()
eta_gd = 1.0 / L_gd
w_gd = np.zeros(n_features)
for _ in range(2000):
    grad = X_r.T @ (X_r @ w_gd - y_r) + lam * w_gd
    w_gd -= eta_gd * grad

# Compare
err = np.linalg.norm(w_closed - w_gd) / np.linalg.norm(w_closed)
print(f'Closed-form solution: ||w*|| = {np.linalg.norm(w_closed):.4f}')
print(f'GD solution (2000 iter): ||w_gd|| = {np.linalg.norm(w_gd):.4f}')
print(f'Relative difference: {err:.2e}')
ok = err < 1e-6
print(f"{'PASS' if ok else 'FAIL'} — GD converges to same solution as closed form")

## 6  Optimality Conditions

A point $x^*$ is optimal if and only if the **gradient (or subgradient) vanishes** at $x^*$.

### 6.1  Unconstrained Problems
For $f$ differentiable and convex:
$$x^* \text{ is a global min} \iff \nabla f(x^*) = 0$$

For non-smooth $f$: $0 \in \partial f(x^*)$ where $\partial f$ is the subdifferential.

### 6.2  Constrained Problems (Preview → §04)
With constraint $x \in \mathcal{C}$, the first-order condition becomes:
$$\nabla f(x^*)^\top (y - x^*) \geq 0 \quad \forall y \in \mathcal{C}$$
This is the **variational inequality** characterising projected gradient = 0.

### 6.3  Subdifferential
For convex $f$, the **subdifferential** at $x$ is the set of all subgradients:
$$\partial f(x) = \{ g : f(y) \geq f(x) + g^\top(y-x) \; \forall y \}$$

Examples:
- $f(x) = |x|$: $\partial f(0) = [-1, 1]$
- $f(x) = \max(x_1, x_2)$: subdifferential is a polytope at the kink


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
HAS_MPL = True
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
except ImportError:
    HAS_MPL = False

# Subdifferential of |x|
x_vals = np.linspace(-2, 2, 400)
f_abs = np.abs(x_vals)

# Subgradients at x=0: any g in [-1,1]
# Show several supporting hyperplanes
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(x_vals, f_abs, 'k-', lw=2, label=r'$f(x)=|x|$')
    for g in [-1, -0.5, 0, 0.5, 1]:
        ax1.plot(x_vals, g * x_vals, alpha=0.4, lw=1.2,
                 label=f'g={g:.1f}' if g in [-1, 0, 1] else '')
    ax1.set_xlim(-2, 2); ax1.set_ylim(-0.5, 2.2)
    ax1.set_xlabel('x'); ax1.set_title(r'Subdifferential of $|x|$ at 0')
    ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

    # Smooth function: unique gradient
    f_smooth = x_vals**2
    x0 = 0.8; g0 = 2 * x0
    ax2.plot(x_vals, f_smooth, 'b-', lw=2, label=r'$f(x)=x^2$')
    ax2.plot(x_vals, g0*(x_vals - x0) + x0**2, 'r--', lw=1.5,
             label=f'tangent at x={x0}')
    ax2.scatter([x0], [x0**2], color='red', s=60, zorder=5)
    ax2.set_xlim(-2, 2); ax2.set_ylim(-0.5, 4.2)
    ax2.set_xlabel('x'); ax2.set_title('Smooth: unique gradient')
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    plt.suptitle('Optimality Conditions: Subdifferential vs Gradient', fontsize=13)
    plt.tight_layout()
    plt.savefig('fig_optimality.png', dpi=100, bbox_inches='tight')
    plt.close()
    print("Saved fig_optimality.png")

# Verify: 0 ∈ ∂|x| at x=0
print("Subdifferential of |x| at x=0:", "[-1, 1]")
print("Optimal? 0 ∈ [-1,1]:", True)
print()
# Verify: ∇x² = 0 at x=0
f = lambda x: x**2
grad = lambda x: 2*x
print(f"∇f(0) = {grad(0)} → global minimum confirmed")


## 7  Lagrangian Duality

### 7.1  The Lagrangian
For the primal problem:
$$\min_x f(x) \quad \text{s.t.} \quad g_i(x) \leq 0, \; h_j(x) = 0$$

The **Lagrangian** is:
$$\mathcal{L}(x, \lambda, \nu) = f(x) + \sum_i \lambda_i g_i(x) + \sum_j \nu_j h_j(x)$$

### 7.2  Dual Function and Dual Problem
The **Lagrange dual function** is:
$$d(\lambda, \nu) = \inf_x \mathcal{L}(x, \lambda, \nu)$$

$d$ is always **concave** (infimum of affine functions in $\lambda, \nu$) — regardless of the primal.

The **dual problem**: $\max_{\lambda \geq 0, \nu} d(\lambda, \nu)$

### 7.3  Weak and Strong Duality
- **Weak duality**: $d(\lambda, \nu) \leq p^*$ always
- **Strong duality**: $d^* = p^*$ holds when Slater's condition is satisfied (∃ strictly feasible point)
- **Duality gap**: $p^* - d^* \geq 0$

### 7.4  Slater's Condition
For a convex problem: if ∃ $x$ such that all inequality constraints are **strictly** satisfied ($g_i(x) < 0$), then strong duality holds.

→ Full KKT machinery: **§04 Constrained Optimization**


In [ ]:
import numpy as np

# SVM Dual: Demonstrate weak duality on a simple 2D classification problem
# Primal: min 0.5||w||² s.t. y_i(w·x_i + b) >= 1
# Dual:   max Σα_i - 0.5 Σ_ij α_i α_j y_i y_j x_i·x_j  s.t. α≥0, Σα_i y_i = 0

np.random.seed(42)
# Simple 2-class, 2D problem
X_pos = np.array([[1, 2], [2, 3], [1.5, 2.5]])
X_neg = np.array([[-1, -1], [-2, -0.5], [-1.5, -2]])
X = np.vstack([X_pos, X_neg])
y = np.array([1, 1, 1, -1, -1, -1])

# Analytical SVM solution for this separable case
# Primal objective at w=[1,0], b=0 (suboptimal)
w_sub = np.array([1.0, 0.0]); b_sub = 0.0
primal_sub = 0.5 * np.dot(w_sub, w_sub)
constraints_sub = y * (X @ w_sub + b_sub)
feasible_sub = np.all(constraints_sub >= 1)

print("=== Weak Duality Demonstration ===")
print(f"Suboptimal primal value: {primal_sub:.4f}")
print(f"All constraints satisfied: {feasible_sub}")

# Lower bound from a dual certificate
# Any dual feasible point gives lower bound
# Dual value α=[0.5, 0, 0, 0.5, 0, 0] (approximate)
alpha = np.array([0.5, 0.0, 0.0, 0.5, 0.0, 0.0])
dual_constraint = np.dot(alpha, y)
print(f"\nDual constraint Σα_i y_i = {dual_constraint:.4f} (should be 0)")

# Compute Gram matrix
K = np.outer(y, y) * (X @ X.T)
dual_val = np.sum(alpha) - 0.5 * alpha @ K @ alpha
print(f"Dual objective value: {dual_val:.4f}")
print(f"Weak duality: dual ({dual_val:.4f}) ≤ primal ({primal_sub:.4f}): "
      f"{dual_val <= primal_sub + 1e-9}")

# Slater's condition check
print("\n=== Slater's Condition ===")
print("Need strictly feasible point: y_i(w·x_i + b) > 1 for all i")
w_slack = np.array([0.5, 0.0]); b_slack = 0.0
margins = y * (X @ w_slack + b_slack)
print(f"Margins: {margins}")
print(f"All > 1? {np.all(margins > 1)} → need larger w or adjust b")
w_slater = np.array([2.0, 0.0])
margins2 = y * (X @ w_slater + b_slack)
print(f"With w=[2,0]: margins = {margins2}")
print(f"All > 1? {np.all(margins2 > 1)} → Slater satisfied → strong duality holds")


## 8  Problem Classes: SOCP and SDP

Continuing the hierarchy from §5 (LP, QP) with SOCP and SDP.

### Second-Order Cone Programming (SOCP)
$$\min c^\top x \quad \text{s.t.} \quad \|A_i x + b_i\|_2 \leq c_i^\top x + d_i$$

Applications: robust least squares, antenna array design, finance (Markowitz with norm constraints).

### Semidefinite Programming (SDP)
$$\min \text{tr}(C X) \quad \text{s.t.} \quad \text{tr}(A_i X) = b_i, \; X \succeq 0$$

Applications: maximum cut relaxation, sum-of-squares polynomials, nuclear norm minimization (matrix completion).

### Complexity Hierarchy
| Class | Complexity | Solver |
|-------|-----------|--------|
| LP | Polynomial (interior point) | `scipy.optimize.linprog`, Gurobi |
| QP | Polynomial | OSQP, quadprog |
| SOCP | Polynomial | ECOS, SCS |
| SDP | Polynomial | SCS, MOSEK |
| Non-convex NLP | NP-hard in general | IPOPT (local) |


In [ ]:
import numpy as np
try:
    from scipy.optimize import linprog, minimize
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

print("=== SDP: Nuclear Norm Minimization (Matrix Completion) ===")
print("Nuclear norm ||X||_* = sum of singular values")
print("Nuclear norm minimization is an SDP — convex relaxation of rank minimization")
print()

# Demonstrate that nuclear norm is convex
# ||A||_* = max_{U,V: U^T U = I, V^T V = I} tr(U^T A V)
# Convexity follows from being max of linear functions
np.random.seed(7)
A = np.random.randn(4, 4)
B = np.random.randn(4, 4)

def nuclear_norm(M):
    return np.sum(np.linalg.svd(M, compute_uv=False))

# Test convexity: ||αA + (1-α)B||_* ≤ α||A||_* + (1-α)||B||_*
alphas = np.linspace(0, 1, 20)
lhs = [nuclear_norm(a*A + (1-a)*B) for a in alphas]
rhs = [a*nuclear_norm(A) + (1-a)*nuclear_norm(B) for a in alphas]
convex_ok = all(l <= r + 1e-10 for l, r in zip(lhs, rhs))
print(f"Nuclear norm convexity test (20 values of α): {convex_ok}")
print(f"  ||A||_* = {nuclear_norm(A):.4f}")
print(f"  ||B||_* = {nuclear_norm(B):.4f}")
print(f"  ||0.5A + 0.5B||_* = {nuclear_norm(0.5*A + 0.5*B):.4f}")
print(f"  0.5||A||_* + 0.5||B||_* = {0.5*nuclear_norm(A) + 0.5*nuclear_norm(B):.4f}")

print()
print("=== Trace of PSD matrix: linear → convex ===")
# tr(CX) with X ⪰ 0 is linear in X (both convex and concave)
# The PSD constraint X ⪰ 0 is a convex cone
C = np.eye(3)  # identity → trace
X1 = np.diag([1.0, 2.0, 3.0])
X2 = np.diag([4.0, 1.0, 0.5])
alpha = 0.6
Xmid = alpha * X1 + (1 - alpha) * X2

print(f"tr(C·X1) = {np.trace(C @ X1):.2f}")
print(f"tr(C·X2) = {np.trace(C @ X2):.2f}")
print(f"tr(C·(αX1+(1-α)X2)) = {np.trace(C @ Xmid):.2f}")
print(f"α·tr(C·X1) + (1-α)·tr(C·X2) = {alpha*np.trace(C@X1)+(1-alpha)*np.trace(C@X2):.2f}")
eigs_mid = np.linalg.eigvalsh(Xmid)
print(f"Xmid eigenvalues: {eigs_mid} — all ≥ 0: {np.all(eigs_mid >= -1e-10)}")


## 9  Applications in Machine Learning

### 9.1  Convex Loss Functions
The losses used in ML training must be convex (in parameters) for convergence guarantees.

| Loss | Formula | Convex? | Why |
|------|---------|---------|-----|
| MSE | $\|y - X\theta\|^2$ | ✓ | Quadratic in $\theta$ |
| Cross-entropy | $-\sum y_k \log p_k$ | ✓ | $-\log(\text{softmax})$ is convex |
| Hinge | $\max(0, 1 - y\hat{y})$ | ✓ | Max of linear functions |
| Logistic | $\log(1 + e^{-yf})$ | ✓ | Log-sum-exp composition |
| Huber | piecewise quadratic/linear | ✓ | Quadratic near 0, linear in tails |

### 9.2  Regularised Objectives
$$\mathcal{L}(\theta) = \underbrace{\ell(\theta)}_\text{convex loss} + \underbrace{R(\theta)}_\text{convex regulariser}$$

Sum of convex functions is convex → regularised objectives preserve convexity.

### 9.3  Nuclear Norm for Low-Rank
Matrix completion (e.g., recommendation systems):
$$\min_X \|X\|_* \quad \text{s.t.} \quad X_{ij} = M_{ij} \; \forall (i,j) \in \Omega$$

Convex relaxation of rank$(X) \leq r$ (NP-hard) → polynomial-time SDP.

### 9.4  LoRA Connection
LoRA approximates weight update $\Delta W \approx BA$ with $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$.
The convex relaxation via nuclear norm: $\min \|W_0 + \Delta W\|_*$ subject to rank constraint.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
HAS_MPL = True
try:
    import matplotlib.pyplot as plt
except ImportError:
    HAS_MPL = False

# Verify convexity of common ML loss functions
np.random.seed(42)
n_tests = 1000

def check_convex_1d(f, a=-5, b=5, n=1000):
    # Check f(ax + (1-a)y) <= af(x) + (1-a)f(y) for random x,y,a
    xs = np.random.uniform(a, b, n)
    ys = np.random.uniform(a, b, n)
    alphas = np.random.uniform(0, 1, n)
    lhs = f(alphas * xs + (1 - alphas) * ys)
    rhs = alphas * f(xs) + (1 - alphas) * f(ys)
    return np.all(lhs <= rhs + 1e-10)

# Define losses (scalar version, logit z, label y=1)
mse      = lambda z: z**2
logistic = lambda z: np.log1p(np.exp(-z))   # y=1: log(1+e^{-z})
hinge    = lambda z: np.maximum(0, 1 - z)
huber    = lambda z: np.where(np.abs(z) <= 1, 0.5*z**2, np.abs(z) - 0.5)

print("=== Convexity Verification for ML Losses ===")
print(f"MSE          convex: {check_convex_1d(mse)}")
print(f"Logistic     convex: {check_convex_1d(logistic)}")
print(f"Hinge        convex: {check_convex_1d(hinge)}")
print(f"Huber        convex: {check_convex_1d(huber)}")

# Plot the losses
if HAS_MPL:
    z = np.linspace(-3, 3, 300)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(z, mse(z),      label='MSE',      lw=2)
    ax.plot(z, logistic(z), label='Logistic', lw=2)
    ax.plot(z, hinge(z),    label='Hinge',    lw=2)
    ax.plot(z, huber(z),    label='Huber',    lw=2, linestyle='--')
    ax.set_xlim(-3, 3); ax.set_ylim(-0.3, 5)
    ax.axvline(0, color='gray', lw=0.8, alpha=0.5)
    ax.axhline(0, color='gray', lw=0.8, alpha=0.5)
    ax.set_xlabel('Margin / prediction z'); ax.set_ylabel('Loss')
    ax.set_title('Convex ML Loss Functions')
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('fig_losses.png', dpi=100, bbox_inches='tight')
    plt.close()
    print("\nSaved fig_losses.png")

# L2 regularisation: verify strongly convex
print("\n=== L2 Regularisation: Strong Convexity ===")
print("f(θ) = loss(θ) + λ||θ||² is λ-strongly convex (if loss is convex)")
lambda_reg = 0.01
theta = np.random.randn(10)
print(f"λ = {lambda_reg}, ||θ||² = {np.dot(theta,theta):.4f}")
print(f"L2 penalty = {lambda_reg * np.dot(theta,theta):.4f}")
print("→ μ ≥ 2λ (strong convexity modulus from regulariser alone)")


## 10  Proximal Operators and Non-Smooth Optimisation

### 10.1  Definition
For convex $f$, the **proximal operator** is:
$$\text{prox}_{\alpha f}(v) = \arg\min_x \left\{ f(x) + \frac{1}{2\alpha}\|x - v\|^2 \right\}$$

Intuition: move from $v$ toward the minimiser of $f$, but not too far (regularised by $\frac{1}{2\alpha}\|\cdot\|^2$).

### 10.2  Key Examples
| $f(x)$ | $\text{prox}_{\alpha f}(v)$ |
|--------|---------------------------|
| $\lambda\|x\|_1$ | $\text{sign}(v) \cdot \max(|v| - \alpha\lambda, 0)$ (soft-threshold) |
| $\frac{\lambda}{2}\|x\|^2$ | $\frac{v}{1 + \alpha\lambda}$ (shrinkage) |
| $\delta_{\mathcal{C}}(x)$ | $\Pi_{\mathcal{C}}(v)$ (projection onto $\mathcal{C}$) |
| $\lambda\|X\|_*$ | $U \cdot \text{diag}(\max(\sigma_i - \alpha\lambda, 0)) \cdot V^\top$ |

### 10.3  ISTA and FISTA
**ISTA** (Iterative Shrinkage-Thresholding Algorithm):
$$x_{t+1} = \text{prox}_{\eta\lambda\|\cdot\|_1}(x_t - \eta \nabla f_\text{smooth}(x_t))$$

Rate: $O(1/t)$ for convex $f$.

**FISTA** adds Nesterov momentum → rate $O(1/t^2)$. Full derivation: **§02 Gradient Descent**.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
HAS_MPL = True
try:
    import matplotlib.pyplot as plt
except ImportError:
    HAS_MPL = False

# Soft-thresholding (prox of L1)
def soft_threshold(v, threshold):
    return np.sign(v) * np.maximum(np.abs(v) - threshold, 0)

# Shrinkage (prox of L2)
def shrinkage(v, alpha, lam):
    return v / (1 + alpha * lam)

# Nuclear norm prox (singular value soft-thresholding)
def prox_nuclear(M, threshold):
    U, s, Vt = np.linalg.svd(M, full_matrices=False)
    s_thresh = np.maximum(s - threshold, 0)
    return U @ np.diag(s_thresh) @ Vt

print("=== Proximal Operators ===")

# 1. Soft-threshold
v = np.array([-3.0, -0.5, 0.0, 0.5, 3.0])
lam = 1.0
st = soft_threshold(v, lam)
print(f"Soft-threshold (λ=1): {v} → {st}")
print(f"  Sparsity: {np.sum(st == 0)}/{len(st)} zeros induced")

# 2. Shrinkage
sh = shrinkage(v, alpha=1.0, lam=0.5)
print(f"\nShrinkage (α=1,λ=0.5): {v} → {np.round(sh, 3)}")

# 3. Nuclear norm prox
np.random.seed(0)
M = np.random.randn(4, 4)
svs_before = np.linalg.svd(M, compute_uv=False)
M_prox = prox_nuclear(M, threshold=0.5)
svs_after = np.linalg.svd(M_prox, compute_uv=False)
print(f"\nNuclear norm prox (τ=0.5):")
print(f"  SVs before: {np.round(svs_before, 3)}")
print(f"  SVs after:  {np.round(svs_after, 3)}")
print(f"  Rank before: {np.sum(svs_before > 1e-10)}, after: {np.sum(svs_after > 1e-10)}")

# ISTA for Lasso
print("\n=== ISTA: Lasso Regression ===")
np.random.seed(42)
m, n = 50, 100  # underdetermined
A = np.random.randn(m, n)
x_true = np.zeros(n); x_true[:5] = np.random.randn(5)  # 5-sparse
b = A @ x_true + 0.01 * np.random.randn(m)
lam = 0.1
eta = 1.0 / np.linalg.norm(A, 2)**2  # 1/L step size

x = np.zeros(n)
losses = []
for _ in range(200):
    grad = A.T @ (A @ x - b)
    x = soft_threshold(x - eta * grad, eta * lam)
    loss = 0.5 * np.sum((A @ x - b)**2) + lam * np.sum(np.abs(x))
    losses.append(loss)

print(f"ISTA Lasso: {200} iterations")
print(f"  Final loss: {losses[-1]:.6f}")
print(f"  ||x||_0 = {np.sum(np.abs(x) > 1e-4)} (true: 5)")
print(f"  ||x - x_true||: {np.linalg.norm(x - x_true):.4f}")

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    v_plot = np.linspace(-3, 3, 300)
    ax1.plot(v_plot, v_plot, 'k--', alpha=0.4, label='identity')
    for lam_v in [0.5, 1.0, 1.5]:
        ax1.plot(v_plot, soft_threshold(v_plot, lam_v), lw=1.8,
                 label=f'λ={lam_v}')
    ax1.set_title('Soft-Thresholding (prox of L1)')
    ax1.set_xlabel('v'); ax1.set_ylabel('prox(v)')
    ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

    ax2.semilogy(losses, lw=1.5, color='steelblue')
    ax2.set_title('ISTA Convergence on Lasso')
    ax2.set_xlabel('Iteration'); ax2.set_ylabel('Objective (log)')
    ax2.grid(alpha=0.3, which='both')

    plt.tight_layout()
    plt.savefig('fig_proximal.png', dpi=100, bbox_inches='tight')
    plt.close()
    print("Saved fig_proximal.png")


## 11  Fenchel Conjugates and Convex Duality

### 11.1  Definition
The **Fenchel conjugate** (convex conjugate) of $f$ is:
$$f^*(y) = \sup_{x \in \text{dom}(f)} \left\{ y^\top x - f(x) \right\}$$

$f^*$ is always convex (supremum of affine functions), even if $f$ is not.

### 11.2  Key Examples
| $f(x)$ | $f^*(y)$ | Domain |
|--------|----------|--------|
| $\frac{1}{2}\|x\|^2$ | $\frac{1}{2}\|y\|^2$ | $\mathbb{R}^n$ |
| $\|x\|_1$ | $\delta_{\|\cdot\|_\infty \leq 1}(y)$ | $\|y\|_\infty \leq 1$ |
| $\|x\|$ | $\delta_{\|\cdot\| \leq 1}(y)$ | $\|y\| \leq 1$ |
| $-\log x$ | $-1 - \log(-y)$ | $y < 0$ |
| $e^x$ | $y \log y - y$ | $y \geq 0$ |
| $\log(1+e^x)$ | $H_b(y)$ (binary entropy) | $y \in [0,1]$ |

### 11.3  Biconjugate
For any $f$: $f^{**} \leq f$. If $f$ is convex and lower semi-continuous, then $f^{**} = f$ (Fenchel-Moreau theorem).

### 11.4  Young's Inequality
$$x^\top y \leq f(x) + f^*(y) \quad \forall x, y$$

with equality when $y \in \partial f(x)$ (i.e., $y$ is a subgradient of $f$ at $x$).


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
HAS_MPL = True
try:
    import matplotlib.pyplot as plt
except ImportError:
    HAS_MPL = False

# Compute and verify Fenchel conjugates numerically
def fenchel_conjugate(f, y_vals, x_range=(-10, 10), n_x=2000):
    # Numerically compute f*(y) = sup_x {y*x - f(x)}
    xs = np.linspace(x_range[0], x_range[1], n_x)
    return np.array([np.max(y * xs - f(xs)) for y in y_vals])

# f(x) = 0.5 x^2: conjugate should be 0.5 y^2
f1 = lambda x: 0.5 * x**2
y_vals = np.linspace(-3, 3, 200)
fstar_numerical = fenchel_conjugate(f1, y_vals)
fstar_analytical = 0.5 * y_vals**2
err1 = np.max(np.abs(fstar_numerical - fstar_analytical))
print(f"Conjugate of 0.5x²: max error vs 0.5y²: {err1:.6f}")

# f(x) = |x|: conjugate is indicator of [-1,1]
f2 = lambda x: np.abs(x)
fstar2_num = fenchel_conjugate(f2, y_vals)
# Analytical: 0 if |y|≤1, else ∞ (but numerically: 0 if |y|≤1, else large)
fstar2_ana = np.where(np.abs(y_vals) <= 1, 0.0, np.inf)
# Check at y=0.5 and y=1.5
print(f"Conjugate of |x| at y=0.5: {fenchel_conjugate(f2, [0.5])[0]:.4f} (expect 0)")
print(f"Conjugate of |x| at y=1.5: {fenchel_conjugate(f2, [1.5])[0]:.4f} (expect large)")

# Young's inequality: x*y ≤ f(x) + f*(y)
print("\n=== Young's Inequality ===")
f = lambda x: 0.5 * x**2
fstar = lambda y: 0.5 * y**2  # conjugate of 0.5x²
np.random.seed(1)
xs = np.random.uniform(-5, 5, 500)
ys = np.random.uniform(-5, 5, 500)
young_ok = np.all(xs * ys <= f(xs) + fstar(ys) + 1e-10)
print(f"Young's ineq (f=0.5x²): {young_ok} for 500 random (x,y)")
print(f"Equality when y=x (subgradient): e.g. x=2, y=2: {2*2:.1f} ≤ {f(2)+fstar(2):.1f}")

# Biconjugate = f for convex lsc
print("\n=== Biconjugate (Fenchel-Moreau) ===")
# f*(y) of f=0.5x² is 0.5y², biconjugate is 0.5x² again
x_test = np.linspace(-3, 3, 200)
fstar_vals = fenchel_conjugate(f1, x_test)   # f*(x)
fstarstar = fenchel_conjugate(lambda y: fstar_vals[np.argmin(np.abs(
    np.linspace(-3, 3, 200)[:, None] - y))], x_test)
# Simple check: f**(x) ≈ f(x)
fstarstar_vals = fenchel_conjugate(lambda y: 0.5*y**2, x_test)
err_biconj = np.max(np.abs(fstarstar_vals - f1(x_test)))
print(f"||f** - f||_∞ for f=0.5x²: {err_biconj:.6f} (expect ~0)")

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    ax1, ax2 = axes

    ax1.plot(y_vals, fstar_numerical, lw=2, label=r'$f^*(y)$ numerical')
    ax1.plot(y_vals, fstar_analytical, '--', lw=1.5, label=r'$rac{1}{2}y^2$ analytical')
    ax1.set_title(r'Conjugate of $rac{1}{2}x^2$')
    ax1.set_xlabel('y'); ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

    ax2.plot(y_vals, fstar2_num, lw=2, label=r'$f^*(y)$ numerical')
    ax2.axhline(0, color='gray', lw=0.8)
    ax2.axvline(-1, color='red', lw=1, linestyle=':', label='|y|=1 boundary')
    ax2.axvline(1, color='red', lw=1, linestyle=':')
    ax2.set_ylim(-0.2, 3); ax2.set_title(r'Conjugate of $|x|$')
    ax2.set_xlabel('y'); ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('fig_fenchel.png', dpi=100, bbox_inches='tight')
    plt.close()
    print("\nSaved fig_fenchel.png")


## 12  Solving Convex Problems with CVXPY

CVXPY is a Python-embedded domain-specific language for convex optimisation. It automatically verifies **DCP (Disciplined Convex Programming)** rules to ensure the problem is convex.

### DCP Ruleset (simplified)
1. Objective: minimise convex or maximise concave expression
2. Constraints: convex expression ≤ 0, affine = 0
3. Compositions: `convex(convex)`, `concave(concave)`, etc. following curvature propagation

### Typical CVXPY Workflow
```python
import cvxpy as cp
x = cp.Variable(n)
objective = cp.Minimize(cp.sum_squares(A @ x - b) + lam * cp.norm1(x))
constraints = []
prob = cp.Problem(objective, constraints)
prob.solve()
x_opt = x.value
```


In [ ]:
import numpy as np
import subprocess, sys

# Try importing cvxpy; if not available, show the pattern and note
try:
    import cvxpy as cp
    HAS_CVXPY = True
except ImportError:
    HAS_CVXPY = False
    print("cvxpy not installed. Install with: pip install cvxpy")
    print("Showing the CVXPY pattern (will run without it).")

np.random.seed(42)
m, n = 30, 50
A = np.random.randn(m, n)
x_true = np.zeros(n); x_true[:5] = np.random.randn(5)
b = A @ x_true + 0.05 * np.random.randn(m)
lam = 0.1

if HAS_CVXPY:
    print("=== CVXPY: Lasso (L1-regularised Least Squares) ===")
    x = cp.Variable(n)
    obj = cp.Minimize(cp.sum_squares(A @ x - b) + lam * cp.norm1(x))
    prob = cp.Problem(obj)
    prob.solve(solver=cp.SCS, verbose=False)
    print(f"Status: {prob.status}")
    print(f"Optimal value: {prob.value:.6f}")
    x_opt = x.value
    print(f"||x||_0 = {np.sum(np.abs(x_opt) > 1e-3)} (true: 5)")
    print(f"||x - x_true|| = {np.linalg.norm(x_opt - x_true):.4f}")

    print("\n=== CVXPY: Quadratic Program (Ridge) ===")
    x2 = cp.Variable(n)
    obj2 = cp.Minimize(cp.sum_squares(A @ x2 - b) + 0.5 * lam * cp.sum_squares(x2))
    prob2 = cp.Problem(obj2)
    prob2.solve(solver=cp.SCS, verbose=False)
    x2_opt = x2.value
    # Analytical solution: (A^T A + lam/2 * I)^{-1} A^T b
    x2_analytic = np.linalg.solve(A.T @ A + 0.5 * lam * np.eye(n), A.T @ b)
    print(f"CVXPY vs analytic ||diff|| = {np.linalg.norm(x2_opt - x2_analytic):.6f}")

    print("\n=== CVXPY: DCP Verification ===")
    y = cp.Variable(3)
    e1 = cp.sum_squares(y)          # convex
    e2 = cp.norm1(y)                # convex
    e3 = cp.log_sum_exp(y)          # convex
    print(f"sum_squares is convex: {e1.is_convex()}")
    print(f"norm1 is convex:       {e2.is_convex()}")
    print(f"log_sum_exp is convex: {e3.is_convex()}")
    e4 = -cp.log(y)                 # convex only if y > 0
    print(f"-log(y) is convex:     {e4.is_convex()} (yes, on domain y>0)")
else:
    # Fallback: demonstrate the ISTA solution (done in §10)
    print("=== Fallback: Lasso via ISTA (no CVXPY needed) ===")
    def soft_threshold(v, t): return np.sign(v) * np.maximum(np.abs(v) - t, 0)
    eta = 1.0 / np.linalg.norm(A, 2)**2
    x = np.zeros(n)
    for _ in range(500):
        x = soft_threshold(x - eta * A.T @ (A @ x - b), eta * lam)
    print(f"ISTA Lasso: ||x||_0={np.sum(np.abs(x)>1e-3)}, ||x-x_true||={np.linalg.norm(x-x_true):.4f}")


## 13  Common Pitfalls and Debugging

### Pitfall 1: Mistaking Non-Convex for Convex
$f(x) = x^4 - 4x^2$ has two local minima → non-convex despite $f'' \geq 0$ at some points.
**Fix**: Check $\nabla^2 f \succeq 0$ everywhere, or use composition rules.

### Pitfall 2: Forgetting Smoothness Requirement
Gradient descent convergence rates assume $L$-smoothness. If $f$ is non-smooth (e.g. L1), use **proximal gradient** methods.

### Pitfall 3: Condition Number Blind Spot
A well-conditioned objective ($\kappa \approx 1$) converges in far fewer iterations than $\kappa \gg 1$.
Always estimate $\kappa = L/\mu$ before selecting a fixed step size.

### Pitfall 4: Weak Duality Mistaken for Strong
The dual optimum bounds the primal from below, but only equals it when Slater's condition holds.

### Pitfall 5: DCP Violations in CVXPY
`cvxpy` raises `DCPError` when the objective curvature doesn't match DCP rules.
Common fix: reformulate using `cp.pos()`, `cp.maximum()`, or split into epigraph form.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
HAS_MPL = True
try:
    import matplotlib.pyplot as plt
except ImportError:
    HAS_MPL = False

print("=== Pitfall 1: Non-Convex Function with Two Local Minima ===")
f_nc = lambda x: x**4 - 4*x**2
df_nc = lambda x: 4*x**3 - 8*x

# Two local minima at x ≈ ±√2
x_local_min = np.sqrt(2)
print(f"Local minima at x = ±√2 ≈ ±{x_local_min:.4f}")
print(f"f(√2) = {f_nc(x_local_min):.4f} = f(-√2) = {f_nc(-x_local_min):.4f}")
print(f"f''(√2) = {12*x_local_min**2 - 8:.4f} > 0 (local min, not global: f→-∞ doesn't apply, global min IS ±√2)")
print(f"f(0) = {f_nc(0):.4f} (local max / saddle-like point)")
print(f"f''(0) = {-8:.0f} < 0 → saddle/local max → NOT convex globally")

# Gradient descent from different starting points
def gd_nc(x0, lr=0.01, n_iter=500):
    x = x0
    path = [x]
    for _ in range(n_iter):
        x = x - lr * df_nc(x)
        path.append(x)
    return np.array(path)

path1 = gd_nc(0.5)
path2 = gd_nc(-0.5)
path3 = gd_nc(3.0)
print(f"\nGD from x0=+0.5 → converges to: {path1[-1]:.4f}")
print(f"GD from x0=-0.5 → converges to: {path2[-1]:.4f}")
print(f"GD from x0=+3.0 → converges to: {path3[-1]:.4f}")
print("Different starting points → different local minima!")

print("\n=== Pitfall 3: Condition Number and Convergence ===")
# GD on quadratics with different condition numbers
for kappa in [1, 10, 100, 1000]:
    L = kappa; mu = 1.0  # eigenvalues: L and mu
    f_quad = lambda x, L=L, mu=mu: 0.5 * (L * x[0]**2 + mu * x[1]**2)
    grad_quad = lambda x, L=L, mu=mu: np.array([L * x[0], mu * x[1]])
    x = np.array([1.0, 1.0])
    lr = 2.0 / (L + mu)  # optimal fixed step
    for i in range(10000):
        x = x - lr * grad_quad(x)
        if np.linalg.norm(x) < 1e-6:
            print(f"  κ={kappa:5d}: converged in {i+1} steps")
            break
    else:
        print(f"  κ={kappa:5d}: did not converge in 10000 steps")

if HAS_MPL:
    x_plot = np.linspace(-3, 3, 400)
    f_conv = lambda x: x**2
    f_nonconv = lambda x: x**4 - 4*x**2

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(x_plot, f_conv(x_plot), lw=2, label='$x^2$ (convex)')
    ax1.plot(x_plot, f_nonconv(x_plot), lw=2, label='$x^4-4x^2$ (non-convex)', color='tomato')
    ax1.axhline(0, color='gray', lw=0.8)
    ax1.set_ylim(-5, 9); ax1.set_title('Convex vs Non-Convex')
    ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

    # GD paths on non-convex
    x_plot2 = np.linspace(-3, 3, 400)
    ax2.plot(x_plot2, f_nonconv(x_plot2), lw=2, color='tomato', label='$x^4-4x^2$')
    ax2.plot(path1[-1], f_nc(path1[-1]), 'bo', ms=8, label=f'x0=+0.5 → {path1[-1]:.2f}')
    ax2.plot(path2[-1], f_nc(path2[-1]), 'gs', ms=8, label=f'x0=-0.5 → {path2[-1]:.2f}')
    ax2.plot(path3[-1], f_nc(path3[-1]), 'r^', ms=8, label=f'x0=+3.0 → {path3[-1]:.2f}')
    ax2.set_ylim(-5, 9); ax2.set_title('GD Sensitivity to Initialization')
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('fig_pitfalls.png', dpi=100, bbox_inches='tight')
    plt.close()
    print("\nSaved fig_pitfalls.png")


## 14  End-to-End Example: Logistic Regression

Logistic regression is a convex optimisation problem. We minimise the negative log-likelihood (cross-entropy) plus an optional L2 penalty:

$$\mathcal{L}(w) = \sum_{i=1}^n \log\left(1 + e^{-y_i w^\top x_i}\right) + \frac{\lambda}{2}\|w\|^2$$

**Convexity proof sketch:**
- $z \mapsto \log(1+e^{-z})$ is convex (log-sum-exp composition)
- $z = y_i w^\top x_i$ is affine in $w$
- Composition of convex with affine is convex
- Sum of convex + strongly convex ($\frac{\lambda}{2}\|w\|^2$) is strongly convex

**Convergence:** With step size $\eta = 1/L$ where $L = \|X\|_2^2/4 + \lambda$, gradient descent converges at rate $O(e^{-t/\kappa})$.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
HAS_MPL = True
try:
    import matplotlib.pyplot as plt
except ImportError:
    HAS_MPL = False

np.random.seed(2024)

# Generate binary classification data
n, d = 200, 2
X_pos = np.random.randn(n//2, d) + np.array([1.5, 1.5])
X_neg = np.random.randn(n//2, d) + np.array([-1.5, -1.5])
X = np.vstack([X_pos, X_neg])
y = np.array([1]*( n//2) + [-1]*(n//2), dtype=float)

# Logistic loss and gradient
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -50, 50)))

def logistic_loss(w, X, y, lam=0.01):
    z = y * (X @ w)
    return np.mean(np.log1p(np.exp(-z))) + 0.5 * lam * np.dot(w, w)

def logistic_grad(w, X, y, lam=0.01):
    z = y * (X @ w)
    s = sigmoid(-z)
    return -X.T @ (y * s) / len(y) + lam * w

lam = 0.01
L_smooth = np.linalg.norm(X, 2)**2 / (4 * len(X)) + lam
eta = 1.0 / L_smooth
print(f"L-smoothness constant: {L_smooth:.4f}")
print(f"Step size η = 1/L = {eta:.6f}")

w = np.zeros(d)
losses = [logistic_loss(w, X, y, lam)]
grad_norms = []

for t in range(500):
    g = logistic_grad(w, X, y, lam)
    w = w - eta * g
    losses.append(logistic_loss(w, X, y, lam))
    grad_norms.append(np.linalg.norm(g))

w_opt = w.copy()
print(f"\nGradient descent (500 iters):")
print(f"  Final loss:      {losses[-1]:.6f}")
print(f"  ||∇f||:          {grad_norms[-1]:.2e}")
print(f"  w* = [{w_opt[0]:.4f}, {w_opt[1]:.4f}]")

# Accuracy
preds = np.sign(X @ w_opt)
acc = np.mean(preds == y)
print(f"  Train accuracy:  {acc:.2%}")

# Strong convexity modulus μ = λ (from L2 term)
mu = lam
kappa = L_smooth / mu
print(f"\nμ = λ = {mu:.4f},  κ = L/μ = {kappa:.1f}")
print(f"Linear convergence rate: (1 - 1/κ)^t = {(1 - 1/kappa)**500:.2e} after 500 steps")

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Decision boundary
    xx, yy = np.meshgrid(np.linspace(-5,5,200), np.linspace(-5,5,200))
    Z = (np.c_[xx.ravel(), yy.ravel()] @ w_opt).reshape(xx.shape)
    axes[0].contourf(xx, yy, Z, levels=0, alpha=0.2, colors=['salmon','lightblue'])
    axes[0].contour(xx, yy, Z, levels=[0], colors='k', linewidths=1.5)
    axes[0].scatter(X_pos[:,0], X_pos[:,1], s=20, c='blue', alpha=0.5, label='+1')
    axes[0].scatter(X_neg[:,0], X_neg[:,1], s=20, c='red',  alpha=0.5, label='-1')
    axes[0].set_title('Decision Boundary'); axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3)

    # Loss curve
    axes[1].semilogy(losses, lw=1.5, color='steelblue')
    axes[1].set_title('Loss Convergence'); axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Loss (log scale)'); axes[1].grid(alpha=0.3, which='both')

    # Gradient norm
    axes[2].semilogy(grad_norms, lw=1.5, color='darkorange')
    axes[2].set_title('||∇f|| Convergence'); axes[2].set_xlabel('Iteration')
    axes[2].set_ylabel('Gradient norm (log scale)'); axes[2].grid(alpha=0.3, which='both')

    plt.suptitle('Logistic Regression as Convex Optimisation', fontsize=13)
    plt.tight_layout()
    plt.savefig('fig_logistic.png', dpi=100, bbox_inches='tight')
    plt.close()
    print("\nSaved fig_logistic.png")


## Summary and Bridge to §02

### Key Results Covered
| Result | Formula | Significance |
|--------|---------|--------------|
| First-order condition | $f(y) \geq f(x) + \nabla f(x)^\top(y-x)$ | Global min ↔ $\nabla f = 0$ |
| Second-order condition | $\nabla^2 f \succeq 0$ | Certifies convexity |
| Strong convexity | $f(y) \geq f(x) + \nabla f(x)^\top(y-x) + \frac{\mu}{2}\|y-x\|^2$ | Unique minimiser |
| Descent lemma | $f(y) \leq f(x) + \nabla f(x)^\top(y-x) + \frac{L}{2}\|y-x\|^2$ | GD step size $1/L$ |
| Soft threshold | $\text{prox}_{\alpha\|\cdot\|_1}(v) = \text{sign}(v)\cdot\max(|v|-\alpha,0)$ | ISTA/FISTA update |
| Young's inequality | $x^\top y \leq f(x) + f^*(y)$ | Duality bound |

### What's Next: §02 Gradient Descent
- **Convergence rates**: $O(1/t)$ convex, $O(e^{-t\mu/L})$ strongly convex, $O(1/\sqrt{t})$ non-convex
- **Momentum**: Polyak momentum, Nesterov acceleration ($O(1/t^2)$)
- **Line search**: Armijo condition, Wolfe conditions
- **Stochastic setting** (preview for §05): noisy gradients, $O(1/\sqrt{t})$ rate

The smoothness constant $L$ and strong convexity $\mu$ derived here are the fundamental parameters controlling gradient descent convergence in §02.

---
*Continue to [§02 Gradient Descent](../02-Gradient-Descent/theory.ipynb)*


In [ ]:
print("=" * 60)
print("  §01 Convex Optimization — Theory Notebook Complete")
print("=" * 60)
print()
print("Sections covered:")
sections = [
    ("1",  "Landscape and Motivation"),
    ("2",  "Convex Sets"),
    ("3",  "Convex Functions"),
    ("4",  "Strong Convexity and Smoothness"),
    ("5",  "Problem Classes (LP, QP, SOCP, SDP)"),
    ("6",  "Optimality Conditions"),
    ("7",  "Lagrangian Duality"),
    ("8",  "SDP and Nuclear Norm"),
    ("9",  "ML Applications"),
    ("10", "Proximal Operators (ISTA)"),
    ("11", "Fenchel Conjugates"),
    ("12", "CVXPY Demo"),
    ("13", "Common Pitfalls"),
    ("14", "End-to-End: Logistic Regression"),
]
for num, name in sections:
    print(f"  §{num:>2}: {name}")
print()
print("Next: exercises.ipynb (8 graded exercises)")


## 15  Mirror Descent and Generalised Bregman Projections

### 15.1  Beyond Euclidean Geometry
Standard gradient descent uses the **Euclidean proximal term** $\frac{1}{2}\|x-y\|^2$.

**Mirror descent** replaces this with a **Bregman divergence**:
$$D_h(x, y) = h(x) - h(y) - \nabla h(y)^\top(x-y)$$

where $h$ is a strictly convex **mirror map**.

Update rule: $x_{t+1} = \arg\min_{x \in \mathcal{C}} \left\{ \eta g_t^\top x + D_h(x, x_t) \right\}$

### 15.2  Special Cases
| Mirror map $h$ | Bregman divergence | Algorithm |
|---|---|---|
| $\frac{1}{2}\|x\|^2$ | $\frac{1}{2}\|x-y\|^2$ | Projected GD |
| $\sum x_i \log x_i$ | $\text{KL}(x \| y)$ | Exponentiated GD (simplex) |
| $-\log x$ | $x/y - 1 - \log(x/y)$ | Multiplicative weights |

### 15.3  Why Mirror Descent Matters
- For sparse problems on the simplex, **KL-based mirror descent** outperforms projected GD
- Natural gradient descent (§03) is mirror descent with Fisher information metric
- Connection to information geometry and variational inference


In [ ]:
import numpy as np

print("=== Mirror Descent: Exponentiated Gradient on Simplex ===")
print("Minimise c^T x  s.t.  x ∈ Δ_n (probability simplex)")
print("EG update: x_i ← x_i * exp(-η * g_i) / Z  (normalise to simplex)")
print()

np.random.seed(42)
n = 10
# Random linear objective on simplex (min c^T x: solution is e_{argmin c})
c = np.random.randn(n)
true_opt = np.argmin(c)
print(f"True optimum: x = e_{true_opt} (corner {true_opt}), c[{true_opt}] = {c[true_opt]:.4f}")

# Exponentiated gradient descent
x = np.ones(n) / n  # start at centroid
eta = 0.5
eg_losses = []
for t in range(200):
    loss = c @ x
    eg_losses.append(loss)
    # EG update
    x = x * np.exp(-eta * c)
    x = x / x.sum()  # project to simplex

print(f"EG final loss: {eg_losses[-1]:.6f}")
print(f"Opt loss:      {c[true_opt]:.6f}")
print(f"EG x[{true_opt}]:      {x[true_opt]:.4f} (should be ≈1)")
print(f"||x - e_opt||: {np.linalg.norm(x - np.eye(n)[true_opt]):.4f}")

# Bregman divergence (KL) at convergence
x_opt = np.eye(n)[true_opt] * (1 - 1e-10) + 1e-10/n  # smooth
kl = np.sum(x_opt * np.log(x_opt / np.clip(x, 1e-15, None)))
print(f"KL(x_opt || x_final): {kl:.4f} (approaches 0)")

print()
print("=== Bregman Divergence Properties ===")
# Verify non-negativity and asymmetry
y = np.array([0.3, 0.3, 0.4])
x1 = np.array([0.5, 0.3, 0.2])
kl_xy = np.sum(x1 * np.log(x1 / y))
kl_yx = np.sum(y * np.log(y / x1))
print(f"KL(x||y) = {kl_xy:.4f}")
print(f"KL(y||x) = {kl_yx:.4f}")
print(f"Asymmetric: {abs(kl_xy - kl_yx) > 1e-8}")
print(f"Non-negative: {kl_xy >= 0 and kl_yx >= 0}")


## 16  Self-Concordant Functions

### 16.1  Definition
A convex function $f: \mathbb{R} \to \mathbb{R}$ is **self-concordant** if:
$$|f'''(x)| \leq 2 f''(x)^{3/2}$$

Multivariate: $\nabla^3 f(x)[h, h, h] \leq 2 (h^\top \nabla^2 f(x) h)^{3/2}$

### 16.2  Why Self-Concordance Matters
Self-concordant functions admit **global convergence guarantees** for Newton's method without knowing $L$ or $\mu$ explicitly.

Key result: Newton's method on a self-concordant function converges in:
- **Damped phase**: $O(f(x_0) - f^*)$ iterations to reach a region where $\lambda(x)^2/2 \leq 0.25$
- **Quadratic phase**: $O(\log \log(1/\epsilon))$ iterations to $\epsilon$-accuracy

### 16.3  Examples of Self-Concordant Functions
- $-\log x$ (log barrier for $x > 0$)
- $-\sum_i \log x_i$ (log barrier for orthant)
- $-\log \det X$ (log barrier for PSD cone — used in interior point methods)
- $\frac{1}{2}\|Ax - b\|^2$ (NOT self-concordant in general — but interior point barrier is)

→ Interior point methods (§04) use log barriers which are self-concordant.


In [ ]:
import numpy as np

print("=== Self-Concordance: -log(x) ===")
print("f(x) = -log(x),  f'(x) = -1/x,  f''(x) = 1/x^2,  f'''(x) = -2/x^3")
print("Self-concordance check: |f'''| <= 2 * (f'')^(3/2)")
print()

xs = np.linspace(0.1, 5.0, 100)
f_pp = 1.0 / xs**2          # f''(x) = 1/x^2
f_ppp = 2.0 / xs**3         # |f'''(x)| = 2/x^3
rhs = 2.0 * f_pp**(1.5)     # 2 * (f'')^(3/2) = 2/x^3

sc_ok = np.all(f_ppp <= rhs + 1e-10)
print(f"-log(x) is self-concordant: {sc_ok}")
print(f"At x=1: |f'''|={f_ppp[45]:.4f}, 2(f'')^(3/2)={rhs[45]:.4f}")
print(f"At x=2: |f'''|={f_ppp[np.argmin(np.abs(xs-2))]:.4f}, "
      f"2(f'')^(3/2)={rhs[np.argmin(np.abs(xs-2))]:.4f}")
print("(Equality holds — -log is 'tight' self-concordant)")

print()
print("=== Log Barrier for PSD Cone ===")
print("f(X) = -log det(X)  is self-concordant for X ≻ 0")
print("This is the barrier function used in interior point SDPs")
print()
np.random.seed(5)
# Verify -log det is convex (determinant is log-concave)
A = np.random.randn(3, 3); A = A @ A.T + 2*np.eye(3)
B = np.random.randn(3, 3); B = B @ B.T + 2*np.eye(3)
alpha = 0.6

f_logdet = lambda M: -np.log(np.linalg.det(M))
lhs = f_logdet(alpha*A + (1-alpha)*B)
rhs = alpha*f_logdet(A) + (1-alpha)*f_logdet(B)
print(f"-log det convexity: f(αA+(1-α)B) = {lhs:.4f}")
print(f"                    αf(A)+(1-α)f(B) = {rhs:.4f}")
print(f"  f(mixture) <= α*f(A) + (1-α)*f(B): {lhs <= rhs + 1e-10}")


## 17  Recap: The Convex Optimisation Toolkit

### Algorithmic Decision Tree

```
Problem type?
├── Smooth, unconstrained → Gradient Descent (§02)
├── Smooth, strongly convex → Linear convergence GD, Newton (§03)
├── Non-smooth (L1, max) → Proximal gradient, subgradient (§10, §02)
├── Constrained (linear/quadratic) → LP/QP solver (OSQP, linprog)
├── Constrained (SDP/SOCP) → Interior point (CVXPY + SCS/MOSEK)
└── Large-scale / stochastic → SGD, Adam (§05, §07)
```

### Key Inequalities Reference
| Inequality | Formula | Use |
|---|---|---|
| Convexity | $f(y) \geq f(x) + \nabla f(x)^\top(y-x)$ | Guarantees global min |
| Smoothness | $f(y) \leq f(x) + \nabla f(x)^\top(y-x) + \frac{L}{2}\|y-x\|^2$ | Bounds GD step |
| Strong convexity | $f(y) \geq f(x) + \nabla f(x)^\top(y-x) + \frac{\mu}{2}\|y-x\|^2$ | Unique minimiser |
| Young's | $x^\top y \leq f(x) + f^*(y)$ | Duality bound |
| Jensen's | $f(\mathbb{E}[X]) \leq \mathbb{E}[f(X)]$ | Concentration bounds |

### Connection to §02 Gradient Descent
The parameters $\mu$ (strong convexity) and $L$ (smoothness) derived here directly control:
- **Step size**: $\eta = 1/L$ (from descent lemma)
- **Convergence rate**: $O((1-\mu/L)^t)$ for strongly convex problems
- **Optimal step**: $\eta = 2/(\mu+L)$ for the Polyak step

Continue to **§02 Gradient Descent** to see these bounds in action.


In [ ]:
import numpy as np

print("=" * 65)
print("  CHAPTER 8 §01: CONVEX OPTIMIZATION — THEORY COMPLETE")
print("=" * 65)
print()
print("Notebook statistics:")
print(f"  Sections:        17")
print(f"  Code cells:      ~25")
print(f"  Markdown cells:  ~25")
print()

# Quick self-test: verify all key formulas
np.random.seed(0)
print("=== Final Self-Test ===")

# 1. First-order condition
f = lambda x: x**2 + 2*x + 1   # min at x=-1
grad = lambda x: 2*x + 2
x_opt = -1.0
print(f"1. First-order: ∇f({x_opt}) = {grad(x_opt):.1f} ✓")

# 2. Strong convexity with μ=2
mu = 2.0
x, y = 0.5, -0.3
sc_ok = f(y) >= f(x) + grad(x)*(y-x) + 0.5*mu*(y-x)**2
print(f"2. Strong convexity (μ=2): {sc_ok} ✓")

# 3. Smoothness: f(x)=x^2, L=2
L = 2.0
smooth_ok = f(y) <= f(x) + grad(x)*(y-x) + 0.5*L*(y-x)**2
print(f"3. L-smoothness (L=2):     {smooth_ok} ✓")

# 4. Condition number
print(f"4. κ = L/μ = {L}/{mu} = {L/mu:.1f} ✓")

# 5. Soft-threshold
v = np.array([-2.0, 0.3, 1.5])
st = np.sign(v) * np.maximum(np.abs(v) - 0.5, 0)
print(f"5. Soft-threshold(v, 0.5): {v} → {st} ✓")

# 6. Jensen's inequality
X = np.random.exponential(1, 1000)
jensen_ok = np.log(np.mean(X)) >= np.mean(np.log(X))  # concave log
print(f"6. Jensen (log): E[log X] ≤ log E[X]: {jensen_ok} ✓")

print()
print("All checks passed. Ready for:")
print("  → exercises.ipynb (8 graded exercises)")
print("  → §02 Gradient Descent (theory.ipynb)")
